In [ ]:
# @title Setting up the environment
!pip install -q "evidently>=0.7,<0.8"
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
import random
import warnings

import nltk
import pandas
import sklearn.feature_extraction.text
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline

warnings.filterwarnings("ignore")

nltk.download("words")
nltk.download("wordnet")
nltk.download("omw-1.4")


def preprocess(train_file, test_file):
  train_X = pandas.read_csv(train_file, index_col="Id")
  test_X = pandas.read_csv(test_file, index_col="Id")

  train_y = train_X.pop("SalePrice")

  all_X = pandas.concat([train_X, test_X])

  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  all_X = all_X.fillna("NA")

  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"Number of missing values: {all_X.isnull().sum().sum()}")

  return (
    all_X.iloc[: train_X.shape[0], :],
    train_y,
    all_X.iloc[train_X.shape[0] :, :],
  )


def download_medicine_reviews() -> tuple[pandas.DataFrame, pandas.DataFrame]:
  """Data source: https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29

  Citation:
    Felix Gräßer, Surya Kallumadi, Hagen Malberg, and Sebastian Zaunseder.
    2018.
    Aspect-Based Sentiment Analysis of Drug Reviews Applying Cross-Domain and Cross-Data Learning.
    In Proceedings of the 2018 International Conference on Digital Health (DH '18).
    ACM, New York, NY, USA, 121-125.
  """
  raw_data = pandas.read_csv(
    "https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_raw.tsv",
    sep="\t",
    index_col=0,
  )
  return raw_data[["drugName", "condition", "review", "rating"]]


def filter_medicine_reviews(df: pandas.DataFrame, condition: str) -> pandas.DataFrame:
  df = df.loc[
    (df["condition"] == condition) & (df["rating"].isin([1, 10])),
    ["review", "rating"],
  ]
  df["is_positive"] = df["rating"].apply(lambda x: 0 if x == 1 else 1)
  return df.drop(columns="rating")


def split_medicine_reviews(
  df: pandas.DataFrame,
) -> tuple[pandas.DataFrame, pandas.DataFrame]:
  X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    df["review"], df["is_positive"], test_size=0.4, random_state=42, shuffle=True
  )

  reference = pandas.DataFrame({"review": X_train, "is_positive": y_train})
  valid = pandas.DataFrame({"review": X_test, "is_positive": y_test})
  return reference, valid

# Monitoring — Data and Concept Drift

## Getting started with the `evidently` library

### Loading the data

In [ ]:
reference_df, _, current_df = preprocess("ames/train.csv", "ames/test.csv")

### Imports

In [ ]:
from evidently import (
  BinaryClassification,
  DataDefinition,
  Dataset,
  Report,
)
from evidently.descriptors import (
  NonLetterCharacterPercentage,
  OOVWordsPercentage,
  SentenceCount,
  TextLength,
  WordCount,
)
from evidently.metrics import DriftedColumnsCount, UniqueValueCount, ValueDrift
from evidently.presets import (
  ClassificationPreset,
  DataDriftPreset,
  ValueStats,
)
from evidently.tests import lt, lte

### Creating a `Dataset` with a `DataDefinition`

Create two `Dataset` objects with `Dataset.from_pandas`: `reference` for `reference_df` and `current` for `current_df`. The default `DataDefinition` is enough: evidently infers each column's type from its values.

**Useful documentation**

- https://docs.evidentlyai.com/docs/library/data_definition

In [ ]:
# Your code here

#### Additional hints

- `Dataset.from_pandas(df, data_definition=DataDefinition())`: one `Dataset` per `DataFrame`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
reference = Dataset.from_pandas(..., data_definition=DataDefinition())
current = ...
```

#### Solution

In [ ]:
reference = Dataset.from_pandas(reference_df, data_definition=DataDefinition())
current = Dataset.from_pandas(current_df, data_definition=DataDefinition())

### Reports

Evidently produces reports, to which tests can be attached.

A report is read: a human studies each column's distributions and scores in it. A test is a condition checked automatically on one of the report's metrics, for instance to stop a pipeline step or to trigger a retraining.

Start with a data drift report, with the `DataDriftPreset` preset and its default parameters, that compares `current` with `reference`.

**Useful documentation**

- https://docs.evidentlyai.com/docs/library/report
- https://docs.evidentlyai.com/metrics/preset_data_drift

In [ ]:
# Your code here

#### Additional hints

- `Report([...])`: a list of metrics or presets
- `report.run(current, reference)`: the recent data first, then the reference; it returns a snapshot, which the notebook displays when it's the cell's last expression

A skeleton to copy into the exercise cell, replacing each `...`:

```python
report = Report([...])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
report = Report([DataDriftPreset()])
snapshot = report.run(current, reference)
snapshot

### Inspecting specific columns

In the dataset we use for these hands-on exercises (AMES), two columns are particularly important: `OverallQual` and `GrLivArea`.

Create a report on these two columns with the `UniqueValueCount` metric for the `OverallQual` column and the `ValueStats` preset for the `GrLivArea` column.

**Useful documentation**

- https://docs.evidentlyai.com/metrics/all_metrics

In [ ]:
# Your code here

#### Additional hints

- `UniqueValueCount(column="OverallQual")`: the number of rows of each value
- `ValueStats(column="GrLivArea")`: a column's statistics (mean, quantiles, missing values…)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
report = Report([UniqueValueCount(column=...), ValueStats(column=...)])
snapshot = report.run(current, reference)
snapshot
```

#### Solution

In [ ]:
report = Report(
  [UniqueValueCount(column="OverallQual"), ValueStats(column="GrLivArea")]
)
snapshot = report.run(current, reference)
snapshot

### Configuring and saving a report

By default, `DataDriftPreset` picks each column's method from its type and the number of rows, and declares the dataset drifted when half of its columns drift. Configure it: the `psi` method for every column, and a dataset drifted as soon as 30% of its columns drift (`drift_share`). Print the number and the share of drifted columns. Is it surprising, given that AMES's training and test sets come from the same sales?

Then save this report in JSON, for a program, and in HTML, for a human.

**Useful documentation**

- https://docs.evidentlyai.com/metrics/customize_data_drift
- https://docs.evidentlyai.com/docs/library/output_formats

In [ ]:
# Your code here

#### Additional hints

- `DataDriftPreset(method="psi", drift_share=0.3)`: `psi` works for numerical and categorical columns, whereas `wasserstein` only applies to numerical ones (`num_method="wasserstein"`, with a `cat_method` for the others)
- `snapshot.dict()["metrics"][0]["value"]`: the preset's first metric, `DriftedColumnsCount`, with the number (`count`) and the share (`share`) of drifted columns
- `snapshot.save_json("data-drift-report.json")`, `snapshot.save_html("data-drift-report.html")`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
report = Report([DataDriftPreset(method=..., drift_share=...)])
snapshot = report.run(current, reference)
print(snapshot.dict()["metrics"][0]["value"])
snapshot.save_json(...)
snapshot.save_html(...)
```

#### Solution

In [ ]:
report = Report([DataDriftPreset(method="psi", drift_share=0.3)])
snapshot = report.run(current, reference)
# No column drifts: both sets are drawn from the same sales
print(snapshot.dict()["metrics"][0]["value"])
snapshot.save_json("data-drift-report.json")
snapshot.save_html("data-drift-report.html")

### Tests

Tests are better suited than reading a report to an automated context such as CI/CD. They are attached to a report's metrics, through their `tests` parameter: a condition (`lt`, `lte`, `gt`, `gte`, `eq`…) on the metric's value.

Create a report whose tests check that at most 5 columns drift (`DriftedColumnsCount`), and that the drift score of `OverallQual` and that of `GrLivArea` (`ValueDrift`) stay below 0.05.

**Useful documentation**

- https://docs.evidentlyai.com/docs/library/tests
- https://docs.evidentlyai.com/metrics/all_metrics#data-drift

In [ ]:
# Your code here

#### Additional hints

- `DriftedColumnsCount(tests=[lte(5)])`: a test on the number of drifted columns (`share_tests=[...]` tests their share)
- `ValueDrift(column="OverallQual", tests=[lt(0.05)])`: a test on a column's drift score (here a distance, which the default method picks from the column's type)
- `Report([...], include_tests=True)` would instead attach their default tests to the metrics that have none

A skeleton to copy into the exercise cell, replacing each `...`:

```python
report = Report(
  [
    DriftedColumnsCount(tests=[...]),
    ValueDrift(column=..., tests=[...]),
    ValueDrift(column=..., tests=[...]),
  ]
)
snapshot = report.run(current, reference)
snapshot
```

#### Solution

In [ ]:
report = Report(
  [
    DriftedColumnsCount(tests=[lte(5)]),
    ValueDrift(column="OverallQual", tests=[lt(0.05)]),
    ValueDrift(column="GrLivArea", tests=[lt(0.05)]),
  ]
)
snapshot = report.run(current, reference)
snapshot

### Analyzing results in Python

A pipeline reads the tests' results in Python. From `snapshot.dict()`, compute the percentage of the last report's tests that passed, and print the names of the failed ones.

In [ ]:
# Your code here

#### Additional hints

- `snapshot.dict()["tests"]`: a list of dictionaries, one per test, with the keys `name`, `description` and `status` (`"SUCCESS"` or `"FAIL"`)
- on the previous report, one of the three tests fails

A skeleton to copy into the exercise cell, replacing each `...`:

```python
tests = snapshot.dict()["tests"]
successes = sum(test["status"] == ... for test in tests)
print(...)
print([test["name"] for test in tests if ...])
```

#### Solution

In [ ]:
tests = snapshot.dict()["tests"]
successes = sum(test["status"] == "SUCCESS" for test in tests)
print(f"Tests passed: {successes / len(tests):.0%}")
print("Tests failed:", [test["name"] for test in tests if test["status"] == "FAIL"])

## Applying the `evidently` library to text data

### Loading the data

The data we will use are reviews about medications used to treat several conditions. For now, we will focus on medications used to treat pain, *Pain* in the data.

In [ ]:
raw_data = download_medicine_reviews()
filtered_data = filter_medicine_reviews(raw_data, "Pain")
reference_df, current_df = split_medicine_reviews(filtered_data)

In [ ]:
reference_df

### Training a classification model

In [ ]:
pipeline = sklearn.pipeline.Pipeline(
  [
    (
      "vectorization",
      sklearn.feature_extraction.text.TfidfVectorizer(
        sublinear_tf=True, max_df=0.5, stop_words="english"
      ),
    ),
    (
      "classification",
      sklearn.linear_model.SGDClassifier(
        alpha=0.0001,
        max_iter=50,
        penalty="l1",
        loss="modified_huber",
        random_state=42,
      ),
    ),
  ]
)
pipeline.fit(reference_df["review"].values, reference_df["is_positive"].values)

Create a new column in `reference_df` and `current_df` that contains the predictions:

In [ ]:
reference_df["predictions"] = pipeline.predict(reference_df["review"].values)
current_df["predictions"] = pipeline.predict(current_df["review"].values)

In [ ]:
reference_df

Define two `Dataset` objects (`reference` and `current`) using a `DataDefinition` adapted to the three columns `review`, `is_positive`, and `predictions` in our data: a text column, and a binary classification (its target and its predictions).

**Useful documentation**

- https://docs.evidentlyai.com/docs/library/data_definition

In [ ]:
# Your code here

#### Additional hints

- `DataDefinition(text_columns=[...], classification=[...])`
- `BinaryClassification(target="is_positive", prediction_labels="predictions")`: predictions as labels (0 or 1), not probabilities

A skeleton to copy into the exercise cell, replacing each `...`:

```python
data_definition = DataDefinition(
  text_columns=[...],
  classification=[BinaryClassification(target=..., prediction_labels=...)],
)
reference = Dataset.from_pandas(reference_df, data_definition=data_definition)
current = ...
```

#### Solution

In [ ]:
data_definition = DataDefinition(
  text_columns=["review"],
  classification=[
    BinaryClassification(target="is_positive", prediction_labels="predictions")
  ],
)

reference = Dataset.from_pandas(reference_df, data_definition=data_definition)
current = Dataset.from_pandas(current_df, data_definition=data_definition)

### Classification quality report

Create a classification quality report, with the `ClassificationPreset` preset, that compares the validation data (`current`) with the reference data.

**Useful documentation**

- https://docs.evidentlyai.com/metrics/preset_classification

In [ ]:
# Your code here

#### Additional hints

- `Report([ClassificationPreset()])`, then `report.run(current, reference)` as for drift

A skeleton to copy into the exercise cell, replacing each `...`:

```python
report = Report([...])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
report = Report([ClassificationPreset()])
snapshot = report.run(current, reference)
snapshot

### Detecting a “technical” data drift

We are going to simulate a common event in a processing pipeline: a bug leads to poor-quality processing. Here, we will even simulate two:

- Bug in the cleaning of HTML tags during review preprocessing
- Data ingestion bug causing the data to be in a different language than the training language

In [ ]:
# The machine translation into French of every review about pain: the
# "different language" of the data ingestion bug
translated_reviews = pandas.read_csv(
  "https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_pain_fr.tsv",
  sep="\t",
  index_col="id",
)["review"]


random_html_tags = (
  "<body>",
  "</body>",
  "<html><body>",
  "</body></html>",
  "<h1>",
  "</h1>",
  '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1"></feFuncA></feComponentTransfer><feComposite in2="SourceGraphic" operator="in"></feComposite></filter></defs></svg>',
)


def inject_random_html_tags(s):
  num_tags = 25
  for i in range(num_tags):
    random.seed(i)
    pos = random.choice(range(len(s)))
    s = s[:pos] + random.choice(random_html_tags) + s[pos:]

  return s

In [ ]:
current_disturbed_df = current_df[["review", "is_positive"]].copy()

In [ ]:
disturbed_num = int(len(current_disturbed_df) * 0.5)
random.seed(42)
disturbed_ind = random.sample(list(current_disturbed_df.index), k=disturbed_num)
current_disturbed_df.loc[disturbed_ind[: int(disturbed_num / 10)], "review"] = (
  current_disturbed_df.loc[disturbed_ind[: int(disturbed_num / 10)], "review"].apply(
    inject_random_html_tags
  )
)
translated = disturbed_ind[int(disturbed_num / 10) :]
current_disturbed_df.loc[translated, "review"] = translated_reviews.loc[translated]

In [ ]:
current_disturbed_df["predictions"] = pipeline.predict(
  current_disturbed_df["review"].values
)
current_disturbed = Dataset.from_pandas(
  current_disturbed_df, data_definition=data_definition
)

### Producing a new quality report

Reuse the previous code to analyze the model performance on this degraded data (`current_disturbed`), comparing it this time with the “clean” validation data (`current`).

In [ ]:
# Your code here

#### Additional hints

- the same report, with `current_disturbed` as the recent data and `current` as the reference

A skeleton to copy into the exercise cell, replacing each `...`:

```python
report = Report([ClassificationPreset()])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
report = Report([ClassificationPreset()])
snapshot = report.run(current_disturbed, current)
snapshot

### Analyzing the model’s poor performance

Produce a data drift report limited to the `is_positive` and `predictions` columns and to the text descriptors of the `review` column, that compares `current_disturbed` with `current`.

First define a list `descriptors` of descriptors: the share of characters that aren't letters, the share of out-of-vocabulary words, the number of sentences, the text length and the number of words. Add them to `current` and to `current_disturbed` with their `add_descriptors` method.

**Useful documentation**

- https://docs.evidentlyai.com/docs/library/descriptors
- https://docs.evidentlyai.com/metrics/all_descriptors

In [ ]:
# Your code here

#### Additional hints

- `NonLetterCharacterPercentage`, `OOVWordsPercentage`, `SentenceCount`, `TextLength`, `WordCount`: each takes the text column, `"review"`, and an `alias`, the name of its column; the rest of the notebook uses the aliases `non_letters`, `oov`, `sentence_count`, `text_length` and `word_count`
- `DataDriftPreset(columns=[...])`: the list of the columns to compare, the descriptors' aliases included

A skeleton to copy into the exercise cell, replacing each `...`:

```python
descriptors = [
  NonLetterCharacterPercentage("review", alias="non_letters"),
  OOVWordsPercentage("review", alias=...),
  ...,
]
current.add_descriptors(descriptors)
current_disturbed.add_descriptors(...)
report = Report([DataDriftPreset(columns=["is_positive", "predictions", ...])])
snapshot = report.run(current_disturbed, current)
snapshot
```

#### Solution

In [ ]:
descriptors = [
  NonLetterCharacterPercentage("review", alias="non_letters"),
  OOVWordsPercentage("review", alias="oov"),
  SentenceCount("review", alias="sentence_count"),
  TextLength("review", alias="text_length"),
  WordCount("review", alias="word_count"),
]
descriptor_columns = [
  "non_letters",
  "oov",
  "sentence_count",
  "text_length",
  "word_count",
]

current.add_descriptors(descriptors)
current_disturbed.add_descriptors(descriptors)

report = Report(
  [DataDriftPreset(columns=["is_positive", "predictions", *descriptor_columns])]
)
snapshot = report.run(current_disturbed, current)
snapshot

### Manual inspection of faulty examples

We can observe an increase in long texts and in the presence of out-of-vocabulary (*OOV*) words.

Use the export of descriptors as a dataframe to inspect faulty examples, for instance:

- Reviews with a length greater than 1000 characters
- Reviews with more than 30% out-of-vocabulary words. How does `evidently` define the vocabulary?

In [ ]:
# Your code here

#### Additional hints

- `current_disturbed.as_dataframe()`: the `Dataset`'s columns, plus one per descriptor, named after its alias
- for the vocabulary, `OOVWordsPercentage`'s code is in the installed package (`evidently/legacy/features/OOV_words_percentage_feature.py`), and the setup cell downloads two nltk corpora

A skeleton to copy into the exercise cell, replacing each `...`:

```python
descriptors_df = current_disturbed.as_dataframe()
descriptors_df[descriptors_df["text_length"] > ...]
```

#### Solution

In [ ]:
descriptors_df = current_disturbed.as_dataframe()
descriptors_df

In [ ]:
descriptors_df[descriptors_df["text_length"] > 1000]

In [ ]:
# evidently's vocabulary is nltk's `words` corpus (236,736 English words): each
# word, stripped of what is neither a letter nor a digit and lowercased, is reduced
# to its lemma by the WordNet lemmatizer. A number, a drug name or a French word
# is therefore out of vocabulary.
descriptors_df[descriptors_df["oov"] > 30]

### Gradual data drift

In production, the population rarely changes all at once. Let's simulate batches of 300 reviews in which the share of reviews about medications for depression, *Depression* in the data, rises from 0 to 100%: the model, trained on pain, sees reviews from another domain come in.

In [ ]:
new_content_df = filter_medicine_reviews(raw_data, "Depression")
new_content_df["predictions"] = pipeline.predict(new_content_df["review"].values)

reference.add_descriptors(descriptors)
monitored_columns = ["review", "predictions", *descriptor_columns]


def make_batch(depression_share: float, size: int = 300) -> Dataset:
  n_depression = int(size * depression_share)
  batch_df = pandas.concat(
    [
      current_df.sample(size - n_depression, random_state=42),
      new_content_df.sample(n_depression, random_state=42),
    ]
  )
  return Dataset.from_pandas(
    batch_df, data_definition=data_definition, descriptors=descriptors
  )

### Deciding to retrain

In production, labels arrive long after the predictions (when they arrive at all): the decision to retrain is made on the inputs and the predictions alone. Create a report `drift_report` whose tests check that:

- less than 30% of the monitored columns (`monitored_columns`) drift;
- the text drift score of the `review` column stays below 0.6.

Then write `should_retrain(snapshot)`, which returns `True` if one of the snapshot's tests failed. Try it on `make_batch(0)` and on `make_batch(1)`, against the training data (`reference`), the reference of monitoring in production.

**Useful documentation**

- https://docs.evidentlyai.com/docs/library/tests

In [ ]:
# Your code here


def should_retrain(snapshot) -> bool: ...

#### Additional hints

- `DriftedColumnsCount(columns=monitored_columns, share_tests=[lt(0.3)])`: a test on the share of drifted columns, not on their number
- `ValueDrift(column="review", tests=[lt(0.6)])`: on a text column, evidently trains a classifier to tell the recent texts from the reference ones; its score is that classifier's ROC AUC (0.5: indistinguishable)
- `snapshot.dict()["tests"]` and their `status`, as above; on `make_batch(0)`, `should_retrain` returns `False`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
drift_report = Report(
  [
    DriftedColumnsCount(columns=..., share_tests=[...]),
    ValueDrift(column="review", tests=[...]),
  ]
)


def should_retrain(snapshot) -> bool:
  return any(test["status"] == ... for test in snapshot.dict()["tests"])


print(should_retrain(drift_report.run(make_batch(0), reference)))
print(should_retrain(drift_report.run(make_batch(1), reference)))
```

#### Solution

In [ ]:
drift_report = Report(
  [
    DriftedColumnsCount(columns=monitored_columns, share_tests=[lt(0.3)]),
    ValueDrift(column="review", tests=[lt(0.6)]),
  ]
)


def should_retrain(snapshot) -> bool:
  return any(test["status"] == "FAIL" for test in snapshot.dict()["tests"])


print(should_retrain(drift_report.run(make_batch(0), reference)))
print(should_retrain(drift_report.run(make_batch(1), reference)))

### Dashboard

A dashboard follows these indicators batch after batch. For each share of *Depression* reviews in `[0, 0.2, 0.4, 0.6, 0.8, 1]`, run `drift_report` on `make_batch(share)` against `reference`, and store in a `DataFrame` `dashboard`, one row per batch:

- the share of *Depression* reviews;
- the share of drifted columns and the drift score of `review`;
- the decision of `should_retrain`;
- the model's accuracy on the batch, computed with the labels, which in production would only arrive much later.

Plot the share of drifted columns, the score of `review` and the accuracy against the share of *Depression* reviews. Would the drift have been caught in time to retrain the model?

**Useful documentation**

- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html

In [ ]:
# Your code here

#### Additional hints

- `snapshot.dict()["metrics"]`: `drift_report`'s metrics in their order; `[0]["value"]["share"]` is the share of drifted columns, `[1]["value"]` the score of `review`
- `batch.as_dataframe()`: the batch's `predictions` and `is_positive` columns, whose mean equality is the accuracy
- `dashboard.plot(x="depression_share", y=[...], marker="o")`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
rows = []
for share in [0, 0.2, 0.4, 0.6, 0.8, 1]:
  batch = make_batch(share)
  snapshot = drift_report.run(batch, reference)
  metrics = snapshot.dict()["metrics"]
  batch_df = batch.as_dataframe()
  rows.append(
    {
      "depression_share": share,
      "drifted_columns": ...,
      "review_drift": ...,
      "retrain": should_retrain(snapshot),
      "accuracy": (batch_df["predictions"] == ...).mean(),
    }
  )
dashboard = pandas.DataFrame(rows)
dashboard.plot(x="depression_share", y=[...], marker="o")
dashboard
```

#### Solution

In [ ]:
rows = []
for share in [0, 0.2, 0.4, 0.6, 0.8, 1]:
  batch = make_batch(share)
  snapshot = drift_report.run(batch, reference)
  metrics = snapshot.dict()["metrics"]
  batch_df = batch.as_dataframe()
  rows.append(
    {
      "depression_share": share,
      "drifted_columns": metrics[0]["value"]["share"],
      "review_drift": metrics[1]["value"],
      "retrain": should_retrain(snapshot),
      "accuracy": (batch_df["predictions"] == batch_df["is_positive"]).mean(),
    }
  )
dashboard = pandas.DataFrame(rows)
dashboard.plot(
  x="depression_share",
  y=["drifted_columns", "review_drift", "accuracy"],
  marker="o",
)
# The reviews' text gives away the change of domain without waiting for the labels:
# retraining is requested while the accuracy has only dropped a little.
first = dashboard.loc[dashboard["retrain"], "depression_share"].min()
print(f"Retraining requested from {first:.0%} of Depression reviews")
dashboard

### Concept drift

In concept drift, the inputs don't change: their relation to the target, $P(y \mid x)$, does, and the same reviews no longer mean the same thing. Let's simulate it on the validation data: the reviews that contain “work” (*works*, *worked*…) get their label flipped, as if the meaning of that word had turned around for patients.

In [ ]:
concept_df = current_df.copy()
flipped = concept_df["review"].str.contains("work", case=False)
concept_df.loc[flipped, "is_positive"] = 1 - concept_df.loc[flipped, "is_positive"]
concept = Dataset.from_pandas(
  concept_df, data_definition=data_definition, descriptors=descriptors
)
print(f"Flipped labels: {flipped.mean():.0%} of the reviews")

### Detecting concept drift

Run `drift_report` on `concept` against `reference`, as for the batches: what does `should_retrain` decide? Then produce a classification quality report that compares `concept` with the same reviews before the drift, `current`. Which signal reveals this drift, and when would it be available in production?

In [ ]:
# Your code here

#### Additional hints

- `drift_report.run(concept, reference)` and `should_retrain`, as for the batches
- `Report([ClassificationPreset()])` on `concept` and `current`, as above

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(should_retrain(drift_report.run(..., ...)))
report = Report([ClassificationPreset()])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
# The inputs and the predictions are those of `current`, which follow the same
# distribution as the training data: the drift tests pass. Only the classification
# quality, measured once the labels are known, reveals the concept drift.
print(f"should_retrain: {should_retrain(drift_report.run(concept, reference))}")
report = Report([ClassificationPreset()])
snapshot = report.run(concept, current)
snapshot